In [1]:
import cv2
import torch
from depth_anything_v2.dpt import DepthAnythingV2
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib
import threading
import queue
import time
import socket
import cv2
import numpy as np
from ultralytics import YOLO
import math
import keyboard


xFormers not available
xFormers not available


In [ ]:
import cv2
import torch
from depth_anything_v2.dpt import DepthAnythingV2
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib
import threading
import queue
import time
import socket
import cv2
import numpy as np
from ultralytics import YOLO
import math
import keyboard
cmap = matplotlib.colormaps.get_cmap('Spectral_r')


# ESP32-CAM's IP and port for receiving video frames
ESP32_IP = "192.168.1.106"
ESP32_PORT = 12346

# Laptop's IP and port for sending commands
LAPTOP_IP = "0.0.0.0"  # Listen on all interfaces
LAPTOP_PORT = 12345
result_queue = queue.Queue()
is_depth = False
# Function to receive video frames from ESP32-CAM
def receive_frames():
    global is_depth
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server_socket:
        server_socket.bind((LAPTOP_IP, ESP32_PORT))
        server_socket.listen()
        print(f"Listening for ESP32-CAM on {LAPTOP_IP}:{ESP32_PORT}")
        while True:
            print(1)
            conn, addr = server_socket.accept()
            print(2)
            with conn:
                print(f"Connected by {addr}")
                while True:
                    is_depth = False
                    angle_data = conn.recv(2)
                    if not angle_data: break
                    angle = int.from_bytes(angle_data, byteorder='little')
                    # Receive frame length (4 bytes) - ESP32 sends it automatically
                    size_data = conn.recv(4)
                    size = int.from_bytes(size_data, byteorder='little')
                
                    # Receive frame data
                    frame_data = b''
                    while len(frame_data) < size:
                        remaining = size - len(frame_data)
                        frame_data += conn.recv(4096 if remaining > 4096 else remaining)
                    # print(4)
    
                    # Convert the frame data to an image
                    frame = cv2.imdecode(np.frombuffer(frame_data, dtype=np.uint8), cv2.IMREAD_COLOR)
                    
                    # print("frame encoded")
                    if frame is not None:
                      results1 = model1.predict(frame,conf=0.7)
                      results2 = model2.predict(frame,conf=0.1)
                      for result in results2:
                          
                          if len(result.boxes.xyxy)>0:
                                
                            #  for result in results1:
                                boxes =result.boxes.xyxy.cpu().numpy()
                                classes =result.boxes.cls.cpu().numpy()
                                confidences = result.boxes.conf.cpu().numpy()
                                for box, cls,conf in zip(boxes,classes,confidences):
                                    if result.names[int(cls)] == "car":
                                       if is_depth == False:
                                        depth =model.infer_image(frame)  # HxW raw depth map in numpy array format
                                        depth1 = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
                                        depth1 = depth1.astype(np.uint8) 
                                        depth1 = (cmap(depth1)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
                                        is_depth = True
                                       if is_depth == True:
                                        x1, y1, x2, y2 = map(int, box[:4])
                                        x=(x1+(x2-x1)//2)
                                        y=(y1+(y2-y1)//2)
                                        depth_value=depth[y,x]
                                        result_queue.put(depth_value)

                                        cv2.putText(depth1,
                                                    str(depth_value),
                                                    (5, 30),
                                                    cv2.FONT_HERSHEY_SIMPLEX,
                                                    1,
                                                    (0, 255, 0),
                                                    2,
                                                    cv2.LINE_AA)
                                       
                                       cv2.circle(frame, (x,y), 2, (255, 0, 255), cv2.FILLED)
                                       cv2.putText(frame, result.names[int(cls)] +" "+ str(int(conf*100))+"%", (x1, y1),
                                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

                      
                      
                      for result in results1:
                          
                          if len(result.boxes.xyxy)>0:
                                
                                # for result in results1:
                                boxes =result.boxes.xyxy.cpu().numpy()
                                classes =result.boxes.cls.cpu().numpy()
                                confidences = result.boxes.conf.cpu().numpy()
                                for box, cls,conf in zip(boxes,classes,confidences):
                                    if result.names[int(cls)] == "red" or result.names[int(cls)] == "stop":
                                         if is_depth == False:
                                             depth =model.infer_image(frame)  # HxW raw depth map in numpy array format
                                             depth1 = (depth - depth.min()) / (depth.max() - depth.min()) * 255.0
                                             depth1 = depth1.astype(np.uint8) 
                                             depth1 = (cmap(depth1)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)
                                             is_depth = True
                                         if is_depth == True:
                                             x1, y1, x2, y2 = map(int, box[:4])
                                             x=(x1+(x2-x1)//2)
                                             y=(y1+(y2-y1)//2)
                                             depth_value=depth[y,x]
                                             result_queue.put(depth_value)
                                             cv2.putText(depth1,
                                                         str(depth_value),
                                                         (200, 30),
                                                         cv2.FONT_HERSHEY_SIMPLEX,
                                                         1,
                                                         (255, 0, 0),
                                                         2,
                                                         cv2.LINE_AA)
                                         
                                         cv2.circle(frame, (x,y), 2, (255, 0, 255), cv2.FILLED)
                                         cv2.putText(frame, result.names[int(cls)] +" "+ str(int(conf*100))+"%", (x1, y1),
                                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)



                    if is_depth:
                        cv2.imshow("Depth Map", depth1)                    
                    cv2.imshow("Frame", frame)
                    if cv2.waitKey(1) & 0xFF == 27:  # Press ESC to exit
                        cv2.destroyAllWindows()
                        break
                    
                      


# Function to send commands to ESP32-CAM
def send_commands():
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.connect((ESP32_IP, LAPTOP_PORT))
            print(f"Connected to {ESP32_IP}:{LAPTOP_PORT}")
            s.sendall('W'.encode()+b'\n')
            print("Sent 'W' to ESP32-CAM")
        
            while True:
                try:
                    if result_queue.get() > 4:
                        s.sendall('Q'.encode()+b'\n')
                        print(f"Sent 'Q' to ESP32-CAM")
                    else:
                        s.sendall('W'.encode()+b'\n')
                        print(f"Sent 'W' to ESP32-CAM")
                except Exception as e:
                    print(f"Error: {e}")
                    break


def send_commands2():
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s2:
            s2.connect((ESP32_IP, LAPTOP_PORT))
            print(f"Connected to {ESP32_IP}:{LAPTOP_PORT}")
            print("Press any key to send ")
    
            while True:
                try:
                    # Wait for a keypress event
                    event = keyboard.read_event()
                    if event.event_type == keyboard.KEY_DOWN:  # Only send on keydown
                        key = event.name  # Get the key that was pressed
                        s2.sendall('W'.encode()+b'\n')
                        print(f"Sent 'W' to ESP32-CAM")
                except Exception as e:
                    print(f"Error: {e}")
                    break
                            
                    
        

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'
model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'depth_anything_v2_{encoder}.pth', map_location='cpu',weights_only=True))
model = model.to(DEVICE).eval()
model1 = YOLO(r"C:\Users\pc\OneDrive\Desktop\AI\weights\yolov12_traffic_110.pt").cuda()
model2= YOLO(r"C:\Users\pc\OneDrive\Desktop\AI\weights\yolo12s.pt").cuda()

# Start threads for receiving frames and sending 
receive_thread = threading.Thread(target=receive_frames)
send_thread = threading.Thread(target=send_commands)
send_thread2 = threading.Thread(target=send_commands2)

receive_thread.start()
send_thread.start()
send_thread2.start()

receive_thread.join()
send_thread.join()
send_thread2.join()
#wwwwwwwwwwwwwwwwwwwwwww


xFormers not available
xFormers not available


Listening for ESP32-CAM on 0.0.0.0:12346
1


Exception in thread Thread-7 (send_commands2):
Traceback (most recent call last):
  File "c:\Users\pc\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "C:\Users\pc\AppData\Roaming\Python\Python310\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\pc\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\pc\AppData\Local\Temp\ipykernel_8884\2628552076.py", line 175, in send_commands2
TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
Exception in thread Thread-6 (send_commands):
Traceback (most recent call last):
  File "c:\Users\pc\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
